In [1]:
import pandas as pd
import numpy as np
from datetime import datetime
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

import warnings
warnings.filterwarnings('ignore')

## 지표 모니터링을 위한 데이터 시각화
### 1) 데이터 추출
- SQL을 활용해 DB에서 데이터를 추출했다는 시뮬레이션
### 2) 데이터 전처리
- SQL 또는 Python을 활용해 데이터 전처리 & 스택형 데이터로 Transform
### 3) 데이터 저장 또는 적재 (데이터 마트 생성)
- 서버에 가공된 데이터를 적재하거나(Automation), 로컬 환경에 데이터를 저장한 뒤 시각화 대시보드 제작(서버에 다시 적재했다는 가정)

In [2]:
df = pd.read_csv('../data/dashboard_data.csv')
df

,customer_id,join_date,transaction_date
0,4,2020-05-01 00:00:00,2020-05-25
1,5,2021-04-11 00:00:00,2021-06-05
2,5,2021-04-11 00:00:00,2021-08-08
3,6,2020-11-26 00:00:00,2021-01-05
4,7,2020-03-28 00:00:00,2020-09-30
...,...,...,...
375147,599995,2021-06-12 00:00:00,NaN
375148,599996,2020-06-03 00:00:00,NaN
375149,599997,2021-05-31 00:00:00,NaN
375150,599998,2021-09-29 00:00:00,NaN


#### 제품 주요 관찰지표
1) Acquisition: 얼마나 많은 신규 유저들을 획득했는지를 확인할 수 있는 지표
2) Activation: 신규 유저 중 Aha Moment를 경험한 유저 수를 확인하는 지표
3) Conversion: 신규 유저 중 Aha Moment를 경험한 유저의 비중을 확인할 수 있는 지표
4) Retention: 목표로 한 주요 행동을 유저가 하고 있는지를 확인할 수 있는 지표

In [16]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 375152 entries, 0 to 375151
Data columns (total 3 columns):
 #   Column            Non-Null Count   Dtype         
---  ------            --------------   -----         
 0   customer_id       375152 non-null  int64         
 1   join_date         375152 non-null  datetime64[ns]
 2   transaction_date  75152 non-null   object        
dtypes: datetime64[ns](1), int64(1), object(1)
memory usage: 8.6+ MB


In [17]:
# 가입날짜 datetime 포맷으로 변경
df['join_date'] = df['join_date'].apply(lambda x: datetime.strptime(x, "%Y-%m-%d %H:%M:%S"))

TypeError: strptime() argument 1 must be str, not Timestamp

In [18]:
# datetime 포맷 변경을 위해 null 값인 행을 분리 후 포맷 변경하고 다시 합침.
tmp = df[df['transaction_date'].notnull()]
tmp2 = df[df['transaction_date'].isnull()]
tmp['transaction_date'] = tmp['transaction_date'].apply(lambda x: datetime.strptime(x, '%Y-%m-%d'))

df = pd.concat([tmp, tmp2], axis=0)

### Acquisition 산출하기

In [23]:
# acquisition 산출
acquisition = df.groupby('join_date')['customer_id'].nunique().reset_index()
acquisition.columns = ['partition_day', 'acquisition_cnt']
acquisition

,partition_day,acquisition_cnt
0,2020-01-01,518
1,2020-01-02,535
2,2020-01-03,524
3,2020-01-04,544
4,2020-01-05,493
...,...,...
695,2021-11-26,442
696,2021-11-27,475
697,2021-11-28,466
698,2021-11-29,439


### Activation 산출하기

In [26]:
# activation 산출
df.groupby(['customer_id', 'join_date'])['transaction_date'].max().reset_index()
df

,customer_id,join_date,transaction_date
0,4,2020-05-01,2020-05-25
1,5,2021-04-11,2021-06-05
2,5,2021-04-11,2021-08-08
3,6,2020-11-26,2021-01-05
4,7,2020-03-28,2020-09-30
...,...,...,...
375147,599995,2021-06-12,NaT
375148,599996,2020-06-03,NaT
375149,599997,2021-05-31,NaT
375150,599998,2021-09-29,NaT


In [33]:
tmp = df[df['transaction_date'].notnull()]
tmp = tmp.groupby(['customer_id', 'join_date'])['transaction_date'].max().reset_index()
tmp['transaction_yn'] = 1
tmp

,customer_id,join_date,transaction_date,transaction_yn
0,4,2020-05-01,2020-05-25,1
1,5,2021-04-11,2021-08-08,1
2,6,2020-11-26,2021-01-05,1
3,7,2020-03-28,2020-09-30,1
4,9,2020-05-10,2021-01-17,1
...,...,...,...,...
49903,99989,2021-01-09,2021-06-02,1
49904,99994,2020-09-13,2020-10-28,1
49905,99996,2021-04-13,2021-04-19,1
49906,99999,2020-04-03,2020-05-16,1


In [41]:
# 분리하고 1 라벨링 후 다시 병합하고 거래 없는 id는 0 으로 라벨링
tmp2 = pd.merge(df[['customer_id', 'join_date']], tmp[['customer_id', 'transaction_yn']], how='left', on='customer_id')
tmp2.fillna(0, inplace=True)
# 다시 중복 제거
tmp2 = tmp2.groupby('customer_id')[['join_date', 'transaction_yn']].max().reset_index()
tmp2

,customer_id,join_date,transaction_yn
0,4,2020-05-01,1.0
1,5,2021-04-11,1.0
2,6,2020-11-26,1.0
3,7,2020-03-28,1.0
4,9,2020-05-10,1.0
...,...,...,...
349903,599995,2021-06-12,0.0
349904,599996,2020-06-03,0.0
349905,599997,2021-05-31,0.0
349906,599998,2021-09-29,0.0


### 스택 transform

In [47]:
agg_df = tmp2.groupby('join_date').agg({'customer_id':'nunique',
                                        'transaction_yn':'sum'}).reset_index()

agg_df.columns = ['partition_date', 'acquisition', 'activation']
agg_df['partition_month'] = agg_df['partition_date'].dt.to_period('M')
agg_df['conversion'] = agg_df['activation'] / agg_df['acquisition']

# 스택 형태로!!!!(중요)
agg_df = agg_df.melt(id_vars=['partition_month', 'partition_date'], value_vars=['acquisition', 'activation', 'conversion'], var_name='category')

agg_df

,partition_month,partition_date,category,value
0,2020-01,2020-01-01,acquisition,518.000000
1,2020-01,2020-01-02,acquisition,535.000000
2,2020-01,2020-01-03,acquisition,524.000000
3,2020-01,2020-01-04,acquisition,544.000000
4,2020-01,2020-01-05,acquisition,493.000000
...,...,...,...,...
2095,2021-11,2021-11-26,conversion,0.024887
2096,2021-11,2021-11-27,conversion,0.018947
2097,2021-11,2021-11-28,conversion,0.021459
2098,2021-11,2021-11-29,conversion,0.027335


### 코호트 테이블 만들기

In [62]:
# 결제월과 코호트 월 생성

tmp = df[df['transaction_date'].notnull()]

tmp['transaction_month'] = tmp['transaction_date'].dt.to_period('M')
tmp['cohort_month'] = tmp.groupby('customer_id')['transaction_month'].transform('min')
tmp['period'] = (tmp['transaction_month'] - tmp['cohort_month']).apply(lambda x: x.n)

cohort_table = tmp.groupby(['cohort_month', 'period'])['customer_id'].nunique().reset_index()
cohort_table

,cohort_month,period,customer_id
0,2020-01,0,123
1,2020-01,1,4
2,2020-01,2,3
3,2020-01,3,5
4,2020-01,4,5
...,...,...,...
295,2021-10,1,206
296,2021-10,2,198
297,2021-11,0,2712
298,2021-11,1,194


### 
- 실무에서는 아마 저 테이블들을 서버에 넣어두고 데이터에 변동이 생겨도 이런 코드셋을 저장해서 코드를 자동화 시켜두고 일자만 조절하여 대시보드가 실시간 연동되지 않을까?

In [63]:
agg_df.to_excel('acq_act.xlsx', index=False)
cohort_table.to_excel('cohort.xlsx', index=False)